# Fairness with Fairlearn: MLOPs course context

**Goal.** Show that a model with a good accuracy can still treat two groups differently (here men and women), **measure** the gap, **reduce** it, and add an **automatic check** so the gap cannot come back unnoticed.

**Plan**

| Step | Question it answers |
|---|---|
| 0. Vocabulary | What words do I need? |
| 1. Imports | Which tools? |
| 2. Load the data | Which data? Which column do I drop? |
| 3. Missing values | Are there holes in the data? |
| 4. Categories | Who is in the data? |
| 5. Prepare | How do I turn the data into numbers and split it? |
| 6. Logistic regression | What is the model, how does it decide? |
| 7. Measure | Where is the gap? |
| 8. Fix | How do I reduce the gap? |
| 9. Read the results | What do I say about them? |
| 10. Guardrail | How do I stop the gap from coming back? |
| 11. Questions | What may I be asked? |


> **FR - À l'oral** : Chaque étape répond à une question : quelles données ? quel modèle ? où est l'écart ? comment le réduire ? comment éviter qu'il revienne ? »

## 0. Vocabulary (read this first)

| Term | Meaning |
|---|---|
| **Fairness** | Fairlearn looks at the **harm a system causes to people, group by group**. |
| **Allocation harm** (unequal access) | A group gets fewer opportunities: loans, jobs, information. |
| **Quality-of-service harm** (worse service) | The system works less well for one group than for another. |
| **Sensitive attribute** | The characteristic that defines the groups. Here: `sex`. |
| **Accuracy** | Share of correct predictions. Question: *does the model make mistakes?* |
| **Selection rate** | Share of people who get a "yes" (predicted `>50K`). Question: *how many people get a yes?* |
| **Demographic parity difference** | Highest selection rate minus lowest one, across groups. `0` means every group gets the same share of "yes". |
| **Score / threshold** | The model gives each person a score between 0 and 1. The decision is "yes" above a threshold. |

> **FR - À l'oral** : « Pour Fairlearn, l'équité, c'est regarder le tort causé aux personnes, groupe par groupe. L'accuracy dit si le modèle se trompe. Le selection rate dit à combien de personnes il dit oui. Les deux ne racontent pas la même histoire, et c'est tout l'intérêt de la démo. »

*Source: Fairlearn user guide, "Fairness in Machine Learning".*

## 1. Imports

| Library | What we use it for |
|---|---|
| `pandas` | tables of data (`DataFrame`) |
| `fairlearn.datasets.fetch_adult` | downloads the Adult dataset |
| `fairlearn.metrics` | **measure**: `MetricFrame` (any metric, per group), `selection_rate`, `demographic_parity_difference` |
| `fairlearn.postprocessing.ThresholdOptimizer` | **fix**: one decision threshold per group |
| `sklearn` | the model (`LogisticRegression`), `accuracy_score`, `train_test_split`, `StandardScaler` |

> **FR - À l'oral** : « Fairlearn sert à deux choses : mesurer (`metrics`) et corriger (ici `postprocessing`). Tout le reste, c'est du pandas et du scikit-learn classiques. »

In [5]:
import pandas as pd
from fairlearn.datasets import fetch_adult
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Load the data and drop a column

**The dataset.** *Adult* (UCI): **48,842 people, 14 variables** (numeric and categorical). The task is binary: does a person earn **more than 50,000 $ a year**? `fetch_adult` returns an object with `.data` (the variables) and `.target` (the answer, `">50K"` or `"<=50K"`). `as_frame=True` gives pandas tables. The first call downloads the data, then Fairlearn keeps a copy (folder `~/.fairlearn-data`).

**Dropping a column.** `fnlwgt` is a census *sampling weight*: it says how many people a row stands for, it is not something about the person. We drop it with `drop(columns=[...])`. `errors="ignore"` means "no error if the column is not there". You can drop other columns the same way, for example a column that only repeats another one (`education-num` is the numeric version of `education`).

**The target `y`.** `(data.target == ">50K").astype(int)` turns the answer into `1` (earns more than 50K) or `0`.

> **FR - À l'oral** : « Le dataset Adult, ce sont près de 49 000 personnes décrites par 14 variables. On veut prédire si elles gagnent plus de 50 000 dollars. J'enlève `fnlwgt` : c'est un poids d'échantillonnage du recensement, pas une caractéristique de la personne. »

In [6]:
data = fetch_adult(as_frame=True)
df = data.data.drop(columns=["fnlwgt"], errors="ignore")   # census sampling weight, not about the person
y = (data.target == ">50K").astype(int)                    # 1 = earns more than 50K, 0 = otherwise
print(df.shape)                                            # (people, variables)
df.head()                                                  # first 5 rows

(48842, 13)


,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


## 3. Missing values

`df.isna()` marks every empty cell, `.sum()` counts them per column. We keep only the columns where the count is above 0.

**Why it matters.** `LogisticRegression` cannot read an empty cell. Here we keep the rows: for the **text** columns, `get_dummies` (step 5) simply puts `0` in every dummy column of a person whose value is missing. We could also delete the rows (`dropna`) or fill the holes (imputation), but deleting rows can remove more people from one group than from another, which would change the very data we use to check fairness.

**How to read the output.** Left: column name. Right: number of missing cells. Divide by the number of people (`len(df)`) to get a percentage. Expect a few text columns, check which ones in your output.

> **FR - À l'oral** : « J'ai regardé où il manque des valeurs. Je les garde : pour les colonnes texte, une valeur manquante donne simplement zéro partout dans les colonnes 0/1. Supprimer les lignes est possible, mais cela peut retirer plus de personnes d'un groupe que d'un autre. »

In [3]:
missing = df.isna().sum()
missing[missing > 0]                       # only the columns that have missing values

workclass         2799
occupation        2809
native-country     857
dtype: int64

## 4. Distribution of the categories

For every text column (`category` or `object` type) we print the **share of each category**. `normalize=True` gives shares instead of counts, `dropna=False` keeps the missing ones visible, `.head(5)` shows the 5 biggest categories. The last line gives the **real** share of `>50K` for each sex, before any model.

**Why it matters for fairness.**
- If one group is much smaller, the model has fewer examples to learn from for it. Check the share of `sex` in your output (men are the majority).
- The real shares of `>50K` differ between men and women **in the data itself**. *Demographic parity* asks for equal "yes" shares anyway. That is a **choice** of fairness criterion, not a natural law. Another criterion (for example equal error rates) would give another answer.

> **FR - À l'oral** : « Je regarde qui est dans les données : il y a plus d'hommes que de femmes. Et surtout, dans les données réelles, la part de plus de 50K n'est pas la même selon le sexe. Demander la parité démographique, c'est choisir d'égaliser les "oui" quand même : c'est un choix, pas une vérité. »

In [7]:
cats = df.select_dtypes(["category", "object"]).columns            # the text columns
for col in cats:
    print(df[col].value_counts(normalize=True, dropna=False).round(3).head(5), "\n")
print(y.groupby(df["sex"], observed=True).mean().round(3))        # real share of >50K per sex

workclass
Private             0.694
Self-emp-not-inc    0.079
Local-gov           0.064
NaN                 0.057
State-gov           0.041
Name: proportion, dtype: float64 

education
HS-grad         0.323
Some-college    0.223
Bachelors       0.164
Masters         0.054
Assoc-voc       0.042
Name: proportion, dtype: float64 

marital-status
Married-civ-spouse    0.458
Never-married         0.330
Divorced              0.136
Separated             0.031
Widowed               0.031
Name: proportion, dtype: float64 

occupation
Prof-specialty     0.126
Craft-repair       0.125
Exec-managerial    0.125
Adm-clerical       0.115
Sales              0.113
Name: proportion, dtype: float64 

relationship
Husband          0.404
Not-in-family    0.258
Own-child        0.155
Unmarried        0.105
Wife             0.048
Name: proportion, dtype: float64 

race
White                 0.855
Black                 0.096
Asian-Pac-Islander    0.031
Amer-Indian-Eskimo    0.010
Other                 0.008
N

## 5. Prepare the data

1. **`A = df["sex"]`**: the sensitive attribute, kept aside. It is **not** given to the model (we `drop` it from `X`), it is only used to compare groups.
2. **`pd.get_dummies(...)`**: turns each text column into 0/1 columns (`education` becomes `education_Bachelors`, `education_HS-grad`...). `.astype(float)` makes everything a number.
3. **`train_test_split(X, y, A, test_size=0.3, random_state=0, stratify=y)`**: cuts the people into **70 % to train** and **30 % to test**, with the **same rows** for `X`, `y` and `A`. It returns 2 pieces per table, in this order: `X_tr, X_te, y_tr, y_te, A_tr, A_te`. `random_state=0` makes the cut identical on every run. `stratify=y` keeps the same share of `>50K` in both parts. We test on people the model has **never seen**.
4. **`StandardScaler`**: puts every column on the same scale (mean 0, standard deviation 1). It **learns** the mean and deviation on the training part only (`fit`), then applies them to both (`transform`), so the test people do not influence the preparation.

**Important remark.** Even though `sex` is removed from the inputs, other columns can carry the same information (for example a column that says husband or wife). The Fairlearn FAQ makes this point: information is often encoded redundantly across several features, so removing the sensitive attribute does **not** guarantee fairness. The gap we measure in step 7 is the proof.

> **FR - À l'oral** : « Je mets le sexe de côté : le modèle ne le voit pas, je m'en sers seulement pour comparer. Je transforme le texte en colonnes 0/1, je coupe 70 % pour apprendre et 30 % pour tester, et je mets tout à la même échelle. Retirer le sexe ne suffit pas : d'autres colonnes en portent l'information. »

In [14]:
A = df["sex"]                                                # sensitive attribute, kept aside
X = pd.get_dummies(df.drop(columns="sex")).astype(float)     # text columns -> 0/1 columns
X_tr, X_te, y_tr, y_te, A_tr, A_te = train_test_split(
    X, y, A, test_size=0.3, random_state=0, stratify=y)      # 70 % train / 30 % test
scaler = StandardScaler().fit(X_tr)                          # learn mean and deviation on train only
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)  # same scale for both parts

## 6. The model: logistic regression

**How it works** (scikit-learn user guide). Despite its name, it is a **classification** model.
1. It computes a **weighted sum** of the variables: `w1*x1 + w2*x2 + ... + b`. The weights `w` are **learned from the training data**.
2. It passes the sum through the **logistic function** `1 / (1 + exp(-sum))`, an S-shaped curve that squashes any number into a **score between 0 and 1**: the probability of being `>50K`. This is `predict_proba`.
3. The decision `predict` applies a **threshold, 0.5 by default**: above = `1` (yes), below = `0` (no).

**The parameters we chose.**
- `max_iter=5000`: maximum number of iterations the solver may use to converge. The default is 100. A bigger value avoids a "did not converge" warning on our many columns.
- `random_state=42`: fixes the random draws, for reproducible results. According to the scikit-learn documentation it is only used by the solvers `sag`, `saga` and `liblinear`, and has **no effect on the other solvers**. We use the default solver (`lbfgs`), so here it changes nothing. It is harmless, and useful if someone changes the solver later.
- We do **not** set `solver`: the default `lbfgs` is used.
- Regularization is applied by default (`C=1.0`), which keeps the weights from becoming too large.

**The last line of the cell** shows the scores of the first 3 people: two columns, `[P(0), P(1)]`. These scores are exactly what `ThresholdOptimizer` will work on in step 8.

> **FR - À l'oral** : « C'est un modèle simple : une somme pondérée des variables, écrasée en un score entre 0 et 1, puis un seuil à 0,5. Au-dessus, oui ; en dessous, non. Retenez ce seuil, c'est lui que Fairlearn va changer. Le `random_state` n'a pas d'effet avec le solveur par défaut ; je le garde pour la reproductibilité. »

*Sources: scikit-learn user guide (logistic regression) and `LogisticRegression` API page.*

In [9]:
model = LogisticRegression(random_state=42, max_iter=5000).fit(X_tr, y_tr)
pred = model.predict(X_te)                      # 0/1 decisions (default threshold 0.5)
model.predict_proba(X_te[:3]).round(3)          # scores [P(0), P(1)] of the first 3 test people

array([[0.995, 0.005],
       [0.999, 0.001],
       [0.951, 0.049]])

## 7. Measure the gap with `MetricFrame`

`report(name, pred)` is our own helper. It does 3 things:
1. **`MetricFrame(metrics=..., y_true=..., y_pred=..., sensitive_features=...)`** computes each metric on everybody (`.overall`) and **separately for each group** (`.by_group`). With a dictionary of metrics, `.by_group` is a table: one row per sex, one column per metric.
2. **`demographic_parity_difference(...)`** returns the highest selection rate minus the lowest one. `0` = same share of "yes" for every group.
3. It prints the table, the overall accuracy and the gap, and returns the gap.

**How to read the output.**
- `accuracy` column: does the model make mistakes for this group?
- `selection_rate` column: what share of this group gets a "yes"?
- A **high accuracy with a very low selection rate** means the model says "no" most of the time to this group: unequal access hidden behind a good score.

*Example from our first run (default threshold; your numbers may differ slightly): women 7.6 % "yes", men 25.4 % "yes", gap 0.177.*

> **FR - À l'oral** : « `MetricFrame` calcule les mêmes métriques groupe par groupe. Ici, avec le modèle d'origine, la part de "oui" est très différente entre femmes et hommes : c'est l'écart que l'on veut réduire. Regardez qu'une bonne accuracy peut cacher un accès inégal. »

In [10]:
def report(name, pred):
    mf = MetricFrame(metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
                     y_true=y_te, y_pred=pred, sensitive_features=A_te)
    gap = demographic_parity_difference(y_te, pred, sensitive_features=A_te)
    print(f"\n=== {name} ===\n{mf.by_group.round(3)}\n"
          f"Overall accuracy: {mf.overall['accuracy']:.3f} | gap between groups: {gap:.3f}")
    return gap
gap_original = report("Original model", pred)


=== Original model ===
        accuracy  selection_rate
sex                             
Female     0.925           0.077
Male       0.816           0.253
Overall accuracy: 0.853 | gap between groups: 0.177


## 8. Fix the decisions with `ThresholdOptimizer`

**Idea.** Keep the **same model** and change only the **decision rule**: instead of one threshold (0.5) for everybody, use **one threshold per group**.

**How it works** (Fairlearn user guide).
1. It starts from the scores of the trained model.
2. For **each group** it **creates separate thresholds** and applies them to the scores.
3. To choose them, it **generates all possible thresholds** and keeps the **best combination** for the *objective* (accuracy by default) under the *fairness constraint* (`demographic_parity`: the groups get the same share of "yes").
4. At prediction time it may **pick between two thresholds at random**, which is why `predict` takes a `random_state`.

**The arguments.**
- `estimator=model`: the model we already trained.
- `constraints="demographic_parity"`: the fairness rule. Others exist (for example `equalized_odds`).
- `prefit=True`: the model is already trained, **do not retrain** it.
- `predict_method="predict_proba"`: use the **score**, not the 0/1 decision.
- `fit(..., sensitive_features=A_tr)` and `predict(..., sensitive_features=A_te)`: the group is **required** in both.

**Limits to say out loud.**
- It satisfies the constraint **exactly**, and in many cases this comes **at the expense of accuracy** (the guide's words).
- It **needs the sensitive attribute at prediction time**: we must know each person's group when we decide. My own remark, not from the documentation: using the group to decide may be restricted in some domains, so check with legal experts.

> **FR - À l'oral** : « Je ne réentraîne rien. Je garde le même modèle, donc les mêmes scores, et je change seulement la règle de décision : un seuil par groupe au lieu d'un seuil unique. Fairlearn essaie tous les seuils possibles et garde la meilleure combinaison en précision qui donne aux deux groupes la même part de oui. Contrepartie : on doit connaître le groupe au moment de décider. »

*Sources: Fairlearn user guide (postprocessing) and `ThresholdOptimizer` API page.*

In [11]:
fixed = ThresholdOptimizer(estimator=model, constraints="demographic_parity",
                           prefit=True, predict_method="predict_proba")
fixed.fit(X_tr, y_tr, sensitive_features=A_tr)               # finds one threshold per group
pred_fixed = fixed.predict(X_te, sensitive_features=A_te, random_state=0)
gap_fixed = report("Fixed decisions (ThresholdOptimizer)", pred_fixed)


=== Fixed decisions (ThresholdOptimizer) ===
        accuracy  selection_rate
sex                             
Female     0.903           0.145
Male       0.797           0.150
Overall accuracy: 0.832 | gap between groups: 0.005


## 9. Read the results

| | Original model | Fixed decisions |
|---|---|---|
| Women, share of "yes" | 7.7 % | 14.5 % |
| Men, share of "yes" | 25.3 % | 15.0 % |
| **Gap between groups** | **0.177** | **0.005** |
| Overall accuracy | 0.853 | 0.832 |

Before the fix, **7.7 %** of women and **25.3 %** of men got a "yes": a gap of **0.177**. After the fix, **14.5 %** and **15.0 %**: a gap of **0.005**. The overall accuracy went from **0.853** to **0.832**, a cost of about **2.1 points**.

**Conclusions:**
- The "yes" share of women almost **doubled** (7.7 % to 14.5 %). The one of men dropped by about **10 points** (25.3 % to 15.0 %). The gap is almost gone.
- Accuracy went down for **both** groups: women 0.925 to 0.903, men 0.816 to 0.797 (about 2 points each). The decisions changed for many people: this is the **fairness vs accuracy trade-off** mentioned in the Fairlearn guide.
- Guardrail: the original model **fails** (0.177 > 0.10), the fixed decisions **pass** (0.005).
- This is **one** fairness criterion (equal share of "yes"). It does not prove the model is "fair" in every sense.

> **FR - À l'oral** : « Avant : 7,7 % des femmes et 25,3 % des hommes avaient un oui, soit un écart de 0,177. Après : 14,5 % et 15,0 %, écart de 0,005. La précision globale passe de 0,853 à 0,832 : environ 2 points de coût. On a presque supprimé l'écart pour un coût faible, mais on a respecté un critère, pas résolu l'équité en général. »

## 10. Guardrail: an automatic check

A **guardrail** (*garde-fou*) is an **automatic check that blocks when a rule is broken**.

- `LIMIT = 0.10`: the maximum gap we accept. **It is an arbitrary demo value**; in a real project it must be justified with domain experts.
- The loop prints PASS or FAIL for the original model and for the fixed decisions.
- `raise SystemExit(...)` stops the script with an **error code**. In `demo.py`, run by **GitHub Actions** on every push, an error turns the build **red** (a green tick otherwise). A change in the code or in the data that brings the gap back is caught **before** production.
- The check applies to the **fixed decisions**. If it fails: do not just raise the limit, investigate (data change? model change?).

**To show a red build in the demo**: temporarily set `LIMIT = 0.0`, push, take a screenshot, then restore the value.

> **FR - À l'oral** : « Un garde-fou, c'est un contrôle automatique qui bloque quand une règle est violée, comme un test unitaire mais pour l'équité. Si l'écart dépasse la limite, le script sort en erreur et le build GitHub Actions passe au rouge. La limite de 0,10 est arbitraire : en vrai, on la fixe avec des experts métier. »

In [15]:
LIMIT = 0.10                                    # arbitrary demo value
for name, gap in [("Original", gap_original), ("Fixed", gap_fixed)]:
    print(f"{name}: {'PASS' if gap <= LIMIT else 'FAIL'} (gap {gap:.3f}, limit {LIMIT})")
if gap_fixed > LIMIT:
    raise SystemExit("FAIRNESS CHECK FAILED")   # in CI: exit code 1 -> red build

Original: FAIL (gap 0.177, limit 0.1)
Fixed: PASS (gap 0.005, limit 0.1)


## 11. Questions 

| Question | Answer |
|---|---|
| Why not just remove `sex`? | We did remove it from the inputs, and a gap remains: other columns carry the information (Fairlearn FAQ: information is often encoded redundantly). |
| Why drop `fnlwgt`? | It is a census sampling weight, not a characteristic of the person. |
| Why `random_state=42` if it changes nothing? | With the default `lbfgs` solver it has no effect (scikit-learn docs). It is kept for reproducibility if the solver changes. |
| Why `max_iter=5000`? | The default (100) may stop before the model converges on our many columns. |
| Why 0.10? | Arbitrary demo value. In a real project: justified with domain experts. |
| Why did accuracy drop after the fix? | The decisions changed for many people. This is the trade-off mentioned in the Fairlearn guide. |
| Is it legal to decide with the group? | I do not know: it depends on the domain and the country. Check with legal experts. `ThresholdOptimizer` requires the group at prediction time. |
| Which metric should I choose? | It depends on the harm: demographic parity for unequal access, error metrics per group for worse service. |
| Is the fixed model "fair"? | It meets one criterion. Fairness is also sociotechnical. |
| Other tools? | AI Fairness 360 (IBM), Aequitas, What-If Tool, TensorFlow Fairness Indicators. |